# 02 — Factory pattern com Enum, dataclass e Pydantic

**Fábrica** = um ponto único que recebe uma *descrição* ("quero um canal do tipo `slack` com estes parâmetros") e devolve o
**objeto certo**. Em sistemas guiados por configuração, a descrição vem de um arquivo — então a fábrica é a fronteira entre
texto não confiável e objetos do domínio.

Cenário: canais de notificação (`email`, `slack`, `webhook`) declarados em configuração.

```
config (dict) ──► [qual tipo?] ──► [quais parâmetros são válidos?] ──► objeto pronto
                  Enum / tag        dataclass / Pydantic
```

In [1]:
from dataclasses import dataclass, field, fields
from enum import Enum, StrEnum
from typing import Any, ClassVar, Protocol

## 0. O ponto de partida: `if/elif`

In [2]:
@dataclass(frozen=True, slots=True)
class EmailChannel:
    to: str
    subject_prefix: str = "[app]"

    def send(self, message: str) -> str:
        return f"email para {self.to}: {self.subject_prefix} {message}"


@dataclass(frozen=True, slots=True)
class SlackChannel:
    channel: str
    mention: str | None = None

    def send(self, message: str) -> str:
        prefix = f"@{self.mention} " if self.mention else ""
        return f"slack #{self.channel}: {prefix}{message}"


@dataclass(frozen=True, slots=True)
class WebhookChannel:
    url: str
    retries: int = 3

    def send(self, message: str) -> str:
        return f"POST {self.url} (até {self.retries} tentativas): {message}"


def build_channel_v0(spec: dict) -> Any:
    kind = spec["kind"]
    params = {k: v for k, v in spec.items() if k != "kind"}
    if kind == "email":
        return EmailChannel(**params)
    elif kind == "slack":
        return SlackChannel(**params)
    elif kind == "webhook":
        return WebhookChannel(**params)
    raise ValueError(f"tipo desconhecido: {kind}")


print(f"build_channel_v0({{'kind': 'slack', 'channel': 'ops'}}): {build_channel_v0({'kind': 'slack', 'channel': 'ops'})!r}")

build_channel_v0({'kind': 'slack', 'channel': 'ops'}): SlackChannel(channel='ops', mention=None)


Funciona, mas: cada tipo novo exige editar a função (viola aberto/fechado), os nomes válidos estão espalhados em strings, e
nada valida os **tipos** dos parâmetros:

In [3]:
weird = build_channel_v0({"kind": "webhook", "url": 123, "retries": "muitas"})
print(f"weird: {weird!r}")

weird: WebhookChannel(url=123, retries='muitas')


## 1. Registro (dicionário) indexado por um `Enum`

O `Enum` fecha o vocabulário (valor desconhecido falha **na entrada**) e o dicionário substitui o `if/elif`:

In [4]:
class ChannelKind(StrEnum):
    EMAIL = "email"
    SLACK = "slack"
    WEBHOOK = "webhook"


REGISTRY: dict[ChannelKind, type] = {
    ChannelKind.EMAIL: EmailChannel,
    ChannelKind.SLACK: SlackChannel,
    ChannelKind.WEBHOOK: WebhookChannel,
}


def build_channel(spec: dict) -> Any:
    params = dict(spec)
    raw_kind = params.pop("kind")
    try:
        kind = ChannelKind(raw_kind)
    except ValueError:
        raise ValueError(f"kind {raw_kind!r} inválido; use um de {[k.value for k in ChannelKind]}") from None
    return REGISTRY[kind](**params)


print(f"build_channel(...): {build_channel({'kind': 'email', 'to': 'ops@x.com'})!r}")
try:
    build_channel({"kind": "sms", "number": "123"})
except ValueError as ex:
    print(f"ValueError: {ex}")

build_channel(...): EmailChannel(to='ops@x.com', subject_prefix='[app]')
ValueError: kind 'sms' inválido; use um de ['email', 'slack', 'webhook']


Um teste barato garante que **todo** membro do Enum tem fábrica (esquecer de registrar é o bug típico desse padrão):

In [5]:
missing = set(ChannelKind) - set(REGISTRY)
print(f"tipos sem fábrica: {missing}")

tipos sem fábrica: set()


## 2. Registro por decorador

Cada classe se registra onde é definida; o registro deixa de ser uma lista central que alguém esquece de atualizar:

In [6]:
DECORATED: dict[ChannelKind, type] = {}


def register(kind: ChannelKind):
    def decorator(cls: type) -> type:
        if kind in DECORATED:
            raise ValueError(f"{kind} já registrado por {DECORATED[kind].__name__}")
        DECORATED[kind] = cls
        return cls
    return decorator


@register(ChannelKind.EMAIL)
@dataclass(frozen=True, slots=True)
class EmailChannel2:
    to: str

    def send(self, message: str) -> str:
        return f"email para {self.to}: {message}"


try:
    @register(ChannelKind.EMAIL)
    class Duplicate: ...
except ValueError as ex:
    print(f"ValueError: {ex}")
print(f"DECORATED: {DECORATED}")

ValueError: email já registrado por EmailChannel2
DECORATED: {<ChannelKind.EMAIL: 'email'>: <class '__main__.EmailChannel2'>}


## 3. Autorregistro com `__init_subclass__`

A classe base registra as subclasses no momento da definição, a partir de um argumento de classe (`kind=...`):

In [7]:
class Channel:
    registry: ClassVar[dict[ChannelKind, type["Channel"]]] = {}

    def __init_subclass__(cls, *, kind: ChannelKind, **kwargs):
        super().__init_subclass__(**kwargs)
        if kind in Channel.registry:
            raise TypeError(f"{kind} já registrado")
        Channel.registry[kind] = cls
        cls.kind = kind

    @classmethod
    def create(cls, spec: dict) -> "Channel":
        params = dict(spec)
        kind = ChannelKind(params.pop("kind"))
        return cls.registry[kind](**params)

    def send(self, message: str) -> str:
        raise NotImplementedError


@dataclass(frozen=True)
class Email(Channel, kind=ChannelKind.EMAIL):
    to: str

    def send(self, message: str) -> str:
        return f"email para {self.to}: {message}"


@dataclass(frozen=True)
class Slack(Channel, kind=ChannelKind.SLACK):
    channel: str

    def send(self, message: str) -> str:
        return f"slack #{self.channel}: {message}"


print(f"Channel.registry: {Channel.registry}")
print(f"Channel.create(...).send('oi'): {Channel.create({'kind': 'slack', 'channel': 'ops'}).send('oi')}")

Channel.registry: {<ChannelKind.EMAIL: 'email'>: <class '__main__.Email'>, <ChannelKind.SLACK: 'slack'>: <class '__main__.Slack'>}
Channel.create(...).send('oi'): slack #ops: oi


> ⚠️ **Registro por efeito colateral de import:** a subclasse só existe no registro depois que o módulo dela foi **importado**.
> Num projeto real, um `import app.channels` esquecido faz `kind="slack"` "sumir". Por isso o teste "todo membro do Enum tem
> fábrica" do item 1 é ainda mais importante aqui.

## 4. Construtores alternativos (`from_dict`) e chaves desconhecidas

Um `classmethod` que constrói a partir de um mapeamento é a "fábrica de uma classe só". A decisão importante: **o que fazer com
chaves que a classe não conhece?**

In [8]:
@dataclass(frozen=True, slots=True)
class Retry:
    attempts: int = 3
    backoff_s: float = 0.5

    @classmethod
    def from_dict(cls, data: dict, *, strict: bool = True) -> "Retry":
        known = {f.name for f in fields(cls)}
        unknown = set(data) - known
        if unknown and strict:
            raise TypeError(f"chaves desconhecidas para {cls.__name__}: {sorted(unknown)}")
        return cls(**{k: v for k, v in data.items() if k in known})


print(f"Retry.from_dict({{'attempts': 5}}): {Retry.from_dict({'attempts': 5})!r}")
try:
    Retry.from_dict({"attempts": 5, "backof_s": 1})          # erro de digitação
except TypeError as ex:
    print(f"TypeError: {ex}")
print(f"strict=False ignora: {Retry.from_dict({'attempts': 5, 'backof_s': 1}, strict=False)!r}")

Retry.from_dict({'attempts': 5}): Retry(attempts=5, backoff_s=0.5)
TypeError: chaves desconhecidas para Retry: ['backof_s']
strict=False ignora: Retry(attempts=5, backoff_s=0.5)


Ignorar chaves desconhecidas parece tolerante, mas transforma **erro de digitação em default silencioso** (`backof_s` some e o
backoff fica 0.5). Em configuração, o padrão seguro é rejeitar.

## 5. O próprio `Enum` como fábrica

O membro do Enum pode carregar a classe que constrói. Assim vocabulário e fábrica ficam no mesmo lugar e é **impossível** ter um
membro sem classe:

In [9]:
class Kind(Enum):
    EMAIL = ("email", EmailChannel)
    SLACK = ("slack", SlackChannel)
    WEBHOOK = ("webhook", WebhookChannel)

    def __new__(cls, label: str, factory: type):
        member = object.__new__(cls)
        member._value_ = label          # o valor do membro é só o rótulo: Kind("slack") funciona
        member.factory = factory
        return member

    def build(self, **params):
        return self.factory(**params)


print(f"Kind('slack'): {Kind('slack')!r}")
print(f"Kind('slack').build(channel='ops'): {Kind('slack').build(channel='ops')!r}")

Kind('slack'): <Kind.SLACK: 'slack'>
Kind('slack').build(channel='ops'): SlackChannel(channel='ops', mention=None)


**Pegadinha:** classes podem ser valores de membros, mas **funções não** — uma `def` (ou `lambda`) no corpo do Enum vira método,
não membro. Para usar uma função como valor, embrulhe em `enum.member`:

In [10]:
import enum


def make_email(**params):
    return EmailChannel(**params)


class Factories(Enum):
    AS_CLASS = EmailChannel
    AS_FUNCTION = make_email              # NÃO vira membro
    AS_MEMBER = enum.member(make_email)   # vira


print(f"list(Factories.__members__): {list(Factories.__members__)}")
print(f"Factories.AS_FUNCTION is make_email (é só a função, não um membro): {Factories.AS_FUNCTION is make_email}")

list(Factories.__members__): ['AS_CLASS', 'AS_MEMBER']
Factories.AS_FUNCTION is make_email (é só a função, não um membro): True


## 6. Fábrica declarativa: união discriminada do Pydantic

Até aqui a fábrica escolhia a classe **e** o resto ficava sem validação de tipos (item 0). Uma união discriminada faz as duas
coisas: o campo `kind` escolhe o modelo, e o modelo valida os parâmetros. Não há código de fábrica — só declaração.

In [11]:
from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field, HttpUrl, TypeAdapter, ValidationError


class Spec(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)


class EmailSpec(Spec):
    kind: Literal[ChannelKind.EMAIL] = ChannelKind.EMAIL
    to: str = Field(pattern=r"^[^@\s]+@[^@\s]+$")
    subject_prefix: str = "[app]"


class SlackSpec(Spec):
    kind: Literal[ChannelKind.SLACK] = ChannelKind.SLACK
    channel: str = Field(min_length=1)
    mention: str | None = None


class WebhookSpec(Spec):
    kind: Literal[ChannelKind.WEBHOOK] = ChannelKind.WEBHOOK
    url: HttpUrl
    retries: int = Field(default=3, ge=0, le=10)


ChannelSpec = Annotated[EmailSpec | SlackSpec | WebhookSpec, Field(discriminator="kind")]
channels_adapter = TypeAdapter(list[ChannelSpec])

config = [
    {"kind": "email", "to": "ops@example.com"},
    {"kind": "slack", "channel": "alertas", "mention": "plantao"},
    {"kind": "webhook", "url": "https://hooks.example.com/x", "retries": 5},
]
specs = channels_adapter.validate_python(config)
for spec in specs:
    print(f"{type(spec).__name__}: {spec!r}")

EmailSpec: EmailSpec(kind=<ChannelKind.EMAIL: 'email'>, to='ops@example.com', subject_prefix='[app]')
SlackSpec: SlackSpec(kind=<ChannelKind.SLACK: 'slack'>, channel='alertas', mention='plantao')
WebhookSpec: WebhookSpec(kind=<ChannelKind.WEBHOOK: 'webhook'>, url=HttpUrl('https://hooks.example.com/x'), retries=5)


E os erros já chegam com **posição na lista e tipo do canal** — exatamente o que alguém editando o arquivo precisa:

In [12]:
bad_config = [
    {"kind": "webhook", "url": 123, "retries": "muitas"},
    {"kind": "sms", "number": "123"},
    {"kind": "slack", "chanel": "ops"},
]
try:
    channels_adapter.validate_python(bad_config)
except ValidationError as ex:
    for error in ex.errors():
        print(f"{'.'.join(map(str, error['loc']))}: {error['msg']}")

0.webhook.url: URL input should be a string or URL
0.webhook.retries: Input should be a valid integer, unable to parse string as an integer
1: Input tag 'sms' found using 'kind' does not match any of the expected tags: <ChannelKind.EMAIL: 'email'>, <ChannelKind.SLACK: 'slack'>, <ChannelKind.WEBHOOK: 'webhook'>
2.slack.channel: Field required
2.slack.chanel: Extra inputs are not permitted


### Sem campo de tipo? Discriminador por função

Configuração legada às vezes não tem `kind`. Um `Discriminator` com função decide pela **forma** do dado:

In [13]:
from pydantic import Discriminator, Tag


def guess_kind(value: Any) -> str | None:
    if isinstance(value, BaseModel):
        return value.kind.value
    if not isinstance(value, dict):
        return None
    if "kind" in value:
        return value["kind"]
    for key, kind in (("to", "email"), ("channel", "slack"), ("url", "webhook")):
        if key in value:
            return kind
    return None


LegacySpec = Annotated[
    Annotated[EmailSpec, Tag("email")] | Annotated[SlackSpec, Tag("slack")] | Annotated[WebhookSpec, Tag("webhook")],
    Discriminator(guess_kind),
]
legacy = TypeAdapter(list[LegacySpec]).validate_python([{"to": "a@b.com"}, {"channel": "ops"}])
print(f"legacy: {legacy!r}")

legacy: [EmailSpec(kind=<ChannelKind.EMAIL: 'email'>, to='a@b.com', subject_prefix='[app]'), SlackSpec(kind=<ChannelKind.SLACK: 'slack'>, channel='ops', mention=None)]


## 7. Spec (dados) ≠ objeto de runtime (comportamento)

Os `*Spec` do Pydantic descrevem **o que foi configurado**; os `*Channel` (dataclasses) **fazem** o trabalho. Separar os dois
mantém o Pydantic na fronteira e o domínio livre de dependências. A fábrica final só traduz um no outro:

In [14]:
RUNTIME: dict[type[Spec], type] = {EmailSpec: EmailChannel, SlackSpec: SlackChannel, WebhookSpec: WebhookChannel}


def to_runtime(spec: Spec):
    params = spec.model_dump(exclude={"kind"})
    if isinstance(spec, WebhookSpec):
        params["url"] = str(spec.url)
    return RUNTIME[type(spec)](**params)


for channel in map(to_runtime, specs):
    print(f"{channel.send('deploy concluído')}")

email para ops@example.com: [app] deploy concluído
slack #alertas: @plantao deploy concluído
POST https://hooks.example.com/x (até 5 tentativas): deploy concluído


```mermaid
flowchart LR
    A[config: lista de dicts] -->|TypeAdapter + discriminator| B[EmailSpec / SlackSpec / WebhookSpec<br/>validados, imutáveis]
    B -->|to_runtime: registro Spec → classe| C[EmailChannel / SlackChannel / WebhookChannel<br/>dataclasses com comportamento]
    C --> D[send]
```

## 8. Abstract factory: famílias de objetos por ambiente

Quando várias peças precisam ser **coerentes entre si** (armazenamento e cache do mesmo ambiente), a fábrica produz a família inteira:

In [15]:
class Storage(Protocol):
    def save(self, key: str, data: bytes) -> str: ...


class Cache(Protocol):
    def get(self, key: str) -> bytes | None: ...


@dataclass
class LocalStorage:
    root: str = "/tmp/app"

    def save(self, key: str, data: bytes) -> str:
        return f"{self.root}/{key} ({len(data)} bytes)"


@dataclass
class MemoryCache:
    items: dict[str, bytes] = field(default_factory=dict)

    def get(self, key: str) -> bytes | None:
        return self.items.get(key)


@dataclass
class BucketStorage:
    bucket: str

    def save(self, key: str, data: bytes) -> str:
        return f"s3://{self.bucket}/{key} ({len(data)} bytes)"


@dataclass
class RemoteCache:
    url: str

    def get(self, key: str) -> bytes | None:
        return None


class Environment(StrEnum):
    LOCAL = "local"
    CLOUD = "cloud"


@dataclass(frozen=True)
class Infrastructure:
    storage: Storage
    cache: Cache


FAMILIES = {
    Environment.LOCAL: lambda cfg: Infrastructure(LocalStorage(cfg.get("root", "/tmp/app")), MemoryCache()),
    Environment.CLOUD: lambda cfg: Infrastructure(BucketStorage(cfg["bucket"]), RemoteCache(cfg["cache_url"])),
}

for cfg in [{"environment": "local"}, {"environment": "cloud", "bucket": "relatorios", "cache_url": "redis://cache:6379"}]:
    infra = FAMILIES[Environment(cfg["environment"])](cfg)
    print(f"{cfg['environment']}: {infra.storage.save('r.csv', b'abc')} | cache={type(infra.cache).__name__}")

local: /tmp/app/r.csv (3 bytes) | cache=MemoryCache
cloud: s3://relatorios/r.csv (3 bytes) | cache=RemoteCache


## 9. Plugins: registro entre pacotes (*entry points*)

Quando os tipos vêm de **outros pacotes**, o registro vira metadado de instalação. Um pacote declara no `pyproject.toml`:

```toml
[project.entry-points."relatorios.channels"]
sms = "meu_pacote.sms:SmsChannel"
```

e a aplicação descobre sem importar nada explicitamente:

In [16]:
from importlib.metadata import entry_points

discovered = {ep.name: ep.value for ep in entry_points(group="relatorios.channels")}
print(f"plugins instalados no grupo 'relatorios.channels': {discovered}")

plugins instalados no grupo 'relatorios.channels': {}


(Vazio aqui: nenhum pacote instalado declara esse grupo. O mecanismo é o mesmo que o `pytest` usa para carregar plugins.)

## 10. Qual usar?

| técnica | o tipo é escolhido por | os parâmetros são validados por | bom para |
|---|---|---|---|
| `if/elif` | código | nada | protótipo |
| registro + `Enum` | `Enum` (vocabulário fechado) | construtor da classe (aridade, não tipos) | poucos tipos, tudo no mesmo módulo |
| decorador / `__init_subclass__` | registro distribuído | idem | tipos espalhados em módulos; cuidado com import |
| `Enum` que carrega a classe | o próprio membro | idem | vocabulário e fábrica inseparáveis |
| união discriminada (Pydantic) | campo `kind` (ou função) | **o modelo** (tipos, restrições, extras) | **configuração vinda de arquivo** |
| abstract factory | ambiente/perfil | cada fábrica | famílias de objetos coerentes |
| entry points | metadados de pacotes | o plugin | extensões de terceiros |

Para configuração, a combinação vencedora é **união discriminada para validar + registro Spec → classe para construir** (itens 6 e 7).

In [17]:
assert set(ChannelKind) == set(REGISTRY) == set(Channel.registry) | {ChannelKind.WEBHOOK}
assert [type(s) for s in specs] == [EmailSpec, SlackSpec, WebhookSpec]
assert [type(c) for c in map(to_runtime, specs)] == [EmailChannel, SlackChannel, WebhookChannel]
print("verificações: ok")

verificações: ok
